# Data scraper — FBref si StatsBomb

Notebookul extrage date pentru predictii de performanta ofensiva. Fiecare rand din fisierele de iesire reprezinta un jucator intr-un meci. Datele FBref si StatsBomb sunt salvate separat; nu le combinam automat, fiindca acoperirea si definitiile statisticilor difera.

- FBref ofera statistici agregate pe jucator–meci.
- StatsBomb ofera evenimente si lineups; notebookul le agrega intr-un tabel pe jucator–meci.
- La rerulare, cache-urile valide sunt refolosite. Fisierele StatsBomb se salveaza atomic per meci.


In [1]:
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests

# Detecteaza proiectul atat cand Jupyter porneste din radacina, cat si din notebooks/.
CWD = Path.cwd().resolve()
if CWD.name == "notebooks" and (CWD.parent / "README.md").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "notebooks").is_dir() and (CWD / "README.md").exists():
    PROJECT_ROOT = CWD
else:
    candidates = [p for p in (CWD, *CWD.parents) if (p / "notebooks").is_dir() and (p / "README.md").exists()]
    if not candidates:
        raise RuntimeError("Nu am gasit radacina proiectului. Deschide Jupyter din IAPD_project/ sau notebooks/.")
    PROJECT_ROOT = candidates[0]

DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
CACHE_ROOT = DATA_ROOT / "cache"
FBREF_OUTPUT_DIR = RAW_ROOT / "fbref"
FBREF_CACHE_DIR = CACHE_ROOT / "fbref"  # Pastreaza cache-ul existent pentru reluare.
STATSBOMB_OUTPUT_DIR = RAW_ROOT / "statsbomb"
STATSBOMB_CACHE_DIR = CACHE_ROOT / "statsbomb"
SOCCERDATA_RUNTIME_DIR = CACHE_ROOT / "soccerdata_runtime"

for directory in [FBREF_OUTPUT_DIR, FBREF_CACHE_DIR, STATSBOMB_OUTPUT_DIR, STATSBOMB_CACHE_DIR, SOCCERDATA_RUNTIME_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# soccerdata configureaza logurile la import; directionam configul si logurile in cache-ul proiectului.
os.environ["SOCCERDATA_DIR"] = str(SOCCERDATA_RUNTIME_DIR)
import soccerdata as sd

HTTP = requests.Session()
HTTP.headers.update({"User-Agent": "IAPD-player-performance-project/1.0"})

print("Project:", PROJECT_ROOT)
print("Data:", DATA_ROOT)
print("FBref cache:", FBREF_CACHE_DIR)


[11/16/26 01:43:39] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             /Users/laur/Master Data                                                               
                             Science/sem1/IAPD/IAPD_project/notebooks/data/cache/soccerdata_runtime/c              
                             onfig/teamname_replacements.json.                                                     

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:189
                             /Users/laur/Master Data                                                               
                             Science/sem1/IAPD/IAPD_project/notebooks/data/cache/soccerdata_runtime/               
                             config/league_dict.json.                                                              

Project: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project
Data: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project/notebooks/data
FBref cache: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project/notebooks/data/cache/fbref


## Configurare

Incepem cu un singur sezon din Premier League pentru a limita cererile FBref. Daca exportul functioneaza, poti adauga si alte sezoane. Perechea StatsBomb implicita este FIFA World Cup 2022; poti adauga perechi competitie–sezon din catalogul afisat de notebook.


In [2]:
FBREF_LEAGUES = ["ENG-Premier League"]
FBREF_SEASONS = ["2023-2024"]
FBREF_USE_CACHED_MATCHES_ONLY = True  # False cere toate meciurile sezonului.

# FIFA World Cup 2022. Adauga alte perechi din catalog daca ai nevoie de mai multe aparitii.
STATSBOMB_SELECTIONS = [(43, 106)]

RUN_FBREF = True
RUN_STATSBOMB = True
REFRESH_STATSBOMB = False  # True ignora cache-ul StatsBomb si descarca din nou.
MAX_RETRIES = 4
REQUEST_TIMEOUT = 45
REQUEST_PAUSE_SECONDS = 0.15
MIN_ROWS = 1_000
MIN_FEATURES = 10

print("FBref:", FBREF_LEAGUES, FBREF_SEASONS)
print("FBref cached matches only:", FBREF_USE_CACHED_MATCHES_ONLY)
print("StatsBomb selections:", STATSBOMB_SELECTIONS)


FBref: ['ENG-Premier League'] ['2023-2024']
FBref cached matches only: True
StatsBomb selections: [(43, 106)]


In [3]:
STATSBOMB_BASE_URL = "https://raw.githubusercontent.com/hudl/open-data/master/data"
STATSBOMB_NUMERIC_FEATURES = [
    "Min", "Gls", "Ast", "Sh", "SoT", "xG", "CrdY", "CrdR",
    "Passes", "PassesCompleted", "KeyPasses", "Crosses", "Carries",
    "Pressures", "Interceptions", "Recoveries", "Blocks", "Clearances", "FoulsCommitted",
]
FBREF_FEATURE_CANDIDATES = [
    "Min", "Gls", "Ast", "PK", "PKatt", "Sh", "SoT", "CrdY", "CrdR",
    "Fls", "Fld", "Off", "Crs", "TklW", "Int", "OG", "PKwon", "PKcon",
]


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    temporary_path.replace(path)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False, encoding="utf-8-sig")
    temporary_path.replace(path)


def fetch_json(url, cache_path, refresh=False, expected_type=None):
    cache_path = Path(cache_path)
    if cache_path.exists() and not refresh:
        try:
            with cache_path.open(encoding="utf-8") as f:
                cached = json.load(f)
            if expected_type is None or isinstance(cached, expected_type):
                return cached
            print(f"Cache invalid ca structura: {cache_path.name}; descarc din nou.")
        except (OSError, json.JSONDecodeError):
            print(f"Cache invalid: {cache_path.name}; descarc din nou.")

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = HTTP.get(url, timeout=REQUEST_TIMEOUT)
            if response.status_code in {429, 500, 502, 503, 504} and attempt < MAX_RETRIES:
                if response.status_code == 429:
                    try:
                        wait_seconds = float(response.headers.get("Retry-After", 60))
                    except ValueError:
                        wait_seconds = 60
                else:
                    wait_seconds = min(2 ** attempt, 30)
                print(f"HTTP {response.status_code}; reincerc in {wait_seconds:g} secunde.")
                time.sleep(wait_seconds)
                continue
            response.raise_for_status()
            payload = response.json()
            if expected_type is not None and not isinstance(payload, expected_type):
                raise ValueError(f"Raspuns neasteptat de la {url}")
            atomic_write_json(cache_path, payload)
            time.sleep(REQUEST_PAUSE_SECONDS)
            return payload
        except (requests.Timeout, requests.ConnectionError):
            if attempt >= MAX_RETRIES:
                raise
            wait_seconds = min(2 ** attempt, 30)
            print(f"Conexiune intrerupta; reincerc in {wait_seconds} secunde.")
            time.sleep(wait_seconds)
    raise RuntimeError(f"Nu am primit un raspuns valid de la {url}")


def flatten_columns(frame):
    result = frame.copy()
    names = []
    used = {}
    for column in result.columns:
        parts = list(column) if isinstance(column, tuple) else [column]
        parts = [str(part).strip() for part in parts if str(part).strip() not in {"", "nan"} and not str(part).lower().startswith("unnamed")]
        name = parts[-1] if parts else "column"
        used[name] = used.get(name, 0) + 1
        names.append(name if used[name] == 1 else f"{name}__{used[name]}")
    result.columns = names
    return result


def normalize_ids(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


def clock_to_minutes(value, default):
    if value is None or value == "":
        return default
    try:
        minutes, seconds = str(value).split(":", 1)
        return float(minutes) + float(seconds) / 60
    except (TypeError, ValueError):
        return default


## FBref — statistici pe jucator si meci

`Soccerdata` pastreaza paginile HTML in cache-ul proiectului. Implicit, notebookul foloseste paginile de meci deja salvate pentru sezonul ales; schimba `FBREF_USE_CACHED_MATCHES_ONLY = False` pentru a cere intregul sezon. Daca nu exista pagini compatibile in cache, notebookul revine la sezonul complet.


In [4]:
def extract_fbref():
    fbref = sd.FBref(
        leagues=FBREF_LEAGUES,
        seasons=FBREF_SEASONS,
        data_dir=FBREF_CACHE_DIR,
    )

    schedule = flatten_columns(fbref.read_schedule(force_cache=True).reset_index())
    if "game_id" not in schedule.columns or "match_report" not in schedule.columns:
        raise KeyError(f"Nu am gasit game_id/match_report in programul FBref. Coloane: {list(schedule.columns)}")
    eligible_schedule = schedule[schedule["game_id"].notna() & schedule["match_report"].notna()].copy()
    if FBREF_USE_CACHED_MATCHES_ONLY:
        cached_match_ids = {path.stem.removeprefix("match_").lower() for path in FBREF_CACHE_DIR.glob("match_*.html")}
        cached_schedule = eligible_schedule[
            eligible_schedule["game_id"].astype("string").str.lower().isin(cached_match_ids)
        ]
    else:
        cached_schedule = eligible_schedule.iloc[0:0]

    if not cached_schedule.empty:
        selected_match_ids = normalize_ids(cached_schedule["game_id"]).tolist()
        print(f"FBref: folosesc {len(selected_match_ids)} meciuri cu pagina deja in cache din {len(eligible_schedule)} disponibile.")
    else:
        selected_match_ids = normalize_ids(eligible_schedule["game_id"]).tolist()
        print(f"FBref: nu exista pagini compatibile in cache; cer {len(selected_match_ids)} meciuri din sezon.")

    player_stats = fbref.read_player_match_stats(
        stat_type="summary",
        match_id=selected_match_ids,
        force_cache=True,
    ).reset_index()
    if player_stats.empty:
        raise RuntimeError("FBref nu a returnat randuri pentru selectia aleasa.")
    player_stats = flatten_columns(player_stats)
    # FBref numeste campul timpului de joc "min"; il standardizam cu StatsBomb.
    if "min" in player_stats.columns and "Min" not in player_stats.columns:
        player_stats = player_stats.rename(columns={"min": "Min"})

    if "game_id" not in player_stats.columns or "game_id" not in schedule.columns:
        raise KeyError("Nu am gasit game_id in exportul FBref necesar pentru asocierea datei.")
    date_column = next((c for c in schedule.columns if c.lower() == "date"), None)
    if date_column is None:
        raise KeyError(f"Nu am gasit coloana date in programul FBref. Coloane: {list(schedule.columns)}")

    player_stats["game_id"] = normalize_ids(player_stats["game_id"])
    schedule["game_id"] = normalize_ids(schedule["game_id"])
    schedule_lookup = (
        schedule.dropna(subset=["game_id"])[["game_id", date_column]]
        .drop_duplicates("game_id")
        .rename(columns={date_column: "date"})
    )
    player_stats = player_stats.merge(schedule_lookup, on="game_id", how="left", validate="many_to_one")

    if "player" not in player_stats.columns:
        raise KeyError(f"Nu am gasit coloana player. Coloane FBref: {list(player_stats.columns)}")
    player_stats["date"] = pd.to_datetime(player_stats["date"], errors="coerce", utc=True)
    player_stats = player_stats.dropna(subset=["player", "date"]).copy()
    player_stats["source"] = "fbref"

    output_file = FBREF_OUTPUT_DIR / "fbref_player_match_stats.csv"
    atomic_write_csv(player_stats, output_file)

    available_features = [
        col for col in FBREF_FEATURE_CANDIDATES
        if col in player_stats.columns and pd.to_numeric(player_stats[col], errors="coerce").notna().any()
    ]
    print(f"FBref: {len(player_stats):,} randuri; {len(available_features)} campuri statistice gasite.")
    print("Salvat:", output_file)
    if len(player_stats) >= MIN_ROWS and len(available_features) >= MIN_FEATURES:
        print("PASS — datasetul FBref trece pragurile de volum si features.")
    else:
        print("ATENTIE — verifica sezonul si campurile; acest export nu trece inca ambele praguri.")
    return player_stats


## StatsBomb — agregare la nivel de jucator si meci

Open Data pastreaza evenimentele si lineups separat. Functiile de mai jos folosesc lineups pentru a identifica aparitiile si minutele, apoi agrega suturi, goluri, assisturi, xG, pase si alte actiuni pe jucator–meci. Fisierele JSON per meci sunt checkpointuri; CSV-ul se reconstruieste din ele la fiecare rulare.


In [5]:
def _empty_stats():
    return {
        "Gls": 0, "Ast": 0, "Sh": 0, "SoT": 0, "xG": 0.0,
        "CrdY": 0, "CrdR": 0, "Passes": 0, "PassesCompleted": 0,
        "KeyPasses": 0, "Crosses": 0, "Carries": 0, "Pressures": 0,
        "Interceptions": 0, "Recoveries": 0, "Blocks": 0, "Clearances": 0,
        "FoulsCommitted": 0,
    }


def _card_counts(card_name):
    name = str(card_name or "").lower()
    yellow = int("yellow" in name or "second yellow" in name)
    red = int("red" in name or "second yellow" in name)
    return yellow, red


def aggregate_statsbomb_match(match, events, lineups, competition_name, season_name):
    event_by_id = {event.get("id"): event for event in events if event.get("id")}
    event_minutes = [
        float(event.get("minute", 0) or 0) + float(event.get("second", 0) or 0) / 60
        for event in events
        if event.get("period") in {1, 2, 3, 4}
    ]
    match_end = max(event_minutes, default=90.0)

    home = match.get("home_team") or {}
    away = match.get("away_team") or {}
    home_id = home.get("home_team_id")
    away_id = away.get("away_team_id")
    home_name = home.get("home_team_name")
    away_name = away.get("away_team_name")

    players = {}
    features = {}
    lineup_card_players = set()
    for team_lineup in lineups:
        team_id = team_lineup.get("team_id")
        team_name = team_lineup.get("team_name")
        for player in team_lineup.get("lineup", []) or []:
            segments = player.get("positions") or []
            if not segments:
                continue  # jucator convocat, dar fara minute in meci
            player_id = player.get("player_id")
            if player_id is None:
                continue
            minutes_played = 0.0
            for segment in segments:
                start = clock_to_minutes(segment.get("from"), 0.0)
                end = clock_to_minutes(segment.get("to"), match_end)
                minutes_played += max(0.0, end - start)
            first_position = next((seg.get("position") for seg in segments if seg.get("position")), None)
            players[player_id] = {
                "player_id": player_id,
                "player": player.get("player_name"),
                "team_id": team_id,
                "team": team_name,
                "opponent": away_name if team_id == home_id else home_name,
                "venue": "Home" if team_id == home_id else "Away",
                "position": first_position,
                "Min": minutes_played,
            }
            features[player_id] = _empty_stats()

            for card in player.get("cards") or []:
                card_value = card.get("card") if isinstance(card, dict) else card
                yellow, red = _card_counts(card_value)
                features[player_id]["CrdY"] += yellow
                features[player_id]["CrdR"] += red
                if yellow or red:
                    lineup_card_players.add(player_id)

    for event in events:
        event_type = (event.get("type") or {}).get("name")
        player_id = (event.get("player") or {}).get("id")
        if player_id not in features:
            continue
        stats = features[player_id]

        if event_type == "Shot":
            shot = event.get("shot") or {}
            outcome = (shot.get("outcome") or {}).get("name")
            stats["Sh"] += 1
            stats["xG"] += float(shot.get("statsbomb_xg", shot.get("xg", 0)) or 0)
            if outcome in {"Goal", "Saved"}:
                stats["SoT"] += 1
            if outcome == "Goal":
                stats["Gls"] += 1
                key_pass_id = shot.get("key_pass_id")
                key_pass = event_by_id.get(key_pass_id, {})
                assister_id = (key_pass.get("player") or {}).get("id")
                if assister_id in features:
                    features[assister_id]["Ast"] += 1

        elif event_type == "Pass":
            pass_data = event.get("pass") or {}
            stats["Passes"] += 1
            pass_outcome = (pass_data.get("outcome") or {}).get("name")
            if pass_outcome is None or pass_outcome in {"Complete", "Success"}:
                stats["PassesCompleted"] += 1
            if pass_data.get("assisted_shot_id") or pass_data.get("goal_assist"):
                stats["KeyPasses"] += 1
            if pass_data.get("cross"):
                stats["Crosses"] += 1

        elif event_type == "Carry":
            stats["Carries"] += 1
        elif event_type == "Pressure":
            stats["Pressures"] += 1
        elif event_type == "Interception":
            stats["Interceptions"] += 1
        elif event_type == "Ball Recovery":
            stats["Recoveries"] += 1
        elif event_type == "Block":
            stats["Blocks"] += 1
        elif event_type == "Clearance":
            stats["Clearances"] += 1
        elif event_type == "Foul Committed":
            stats["FoulsCommitted"] += 1

        if player_id not in lineup_card_players and event_type in {"Foul Committed", "Bad Behaviour"}:
            card_event = event.get("foul_committed") or event.get("bad_behaviour") or {}
            card_value = card_event.get("card") if isinstance(card_event, dict) else None
            card_name = card_value.get("name") if isinstance(card_value, dict) else card_value
            yellow, red = _card_counts(card_name)
            stats["CrdY"] += yellow
            stats["CrdR"] += red

    competition = match.get("competition") or {}
    season = match.get("season") or {}
    match_id = match.get("match_id")
    match_date = match.get("match_date")

    rows = []
    for player_id, player in players.items():
        row = {
            "source": "statsbomb",
            "competition": competition.get("competition_name", competition_name),
            "season": season.get("season_name", season_name),
            "match_id": match_id,
            "date": match_date,
            **player,
            **features[player_id],
        }
        rows.append(row)
    return rows


In [6]:
def extract_statsbomb():
    catalog_path = STATSBOMB_CACHE_DIR / "competitions.json"
    competitions = fetch_json(
        f"{STATSBOMB_BASE_URL}/competitions.json",
        catalog_path,
        refresh=REFRESH_STATSBOMB,
        expected_type=list,
    )
    catalog = pd.json_normalize(competitions, sep=".")
    atomic_write_csv(catalog, STATSBOMB_OUTPUT_DIR / "competitions.csv")
    display(catalog[[c for c in ["competition_id", "season_id", "competition_name", "season_name", "country_name"] if c in catalog.columns]].head(30))

    all_player_matches = []
    for competition_id, season_id in STATSBOMB_SELECTIONS:
        selected = [
            row for row in competitions
            if int(row["competition_id"]) == int(competition_id) and int(row["season_id"]) == int(season_id)
        ]
        if not selected:
            raise ValueError(f"Perechea StatsBomb {competition_id}:{season_id} nu apare in catalog.")
        competition_name = selected[0]["competition_name"]
        season_name = selected[0]["season_name"]
        selection_dir = STATSBOMB_OUTPUT_DIR / f"competition_{competition_id}_season_{season_id}"
        events_dir = selection_dir / "events"
        lineups_dir = selection_dir / "lineups"
        events_dir.mkdir(parents=True, exist_ok=True)
        lineups_dir.mkdir(parents=True, exist_ok=True)

        matches_path = selection_dir / "matches_raw.json"
        matches = fetch_json(
            f"{STATSBOMB_BASE_URL}/matches/{competition_id}/{season_id}.json",
            matches_path,
            refresh=REFRESH_STATSBOMB,
            expected_type=list,
        )
        print(f"{competition_name} {season_name}: {len(matches)} meciuri")
        failed_matches = []

        for index, match in enumerate(matches, start=1):
            match_id = match["match_id"]
            events_path = events_dir / f"{match_id}.json"
            lineups_path = lineups_dir / f"{match_id}.json"
            try:
                events = fetch_json(
                    f"{STATSBOMB_BASE_URL}/events/{match_id}.json",
                    events_path,
                    refresh=REFRESH_STATSBOMB,
                    expected_type=list,
                )
                lineups = fetch_json(
                    f"{STATSBOMB_BASE_URL}/lineups/{match_id}.json",
                    lineups_path,
                    refresh=REFRESH_STATSBOMB,
                    expected_type=list,
                )
                all_player_matches.extend(
                    aggregate_statsbomb_match(match, events, lineups, competition_name, season_name)
                )
                print(f"{index}/{len(matches)} — match {match_id}: checkpoint incarcat")
            except Exception as exc:
                failed_matches.append({"match_id": match_id, "error": str(exc)})
                print(f"Avertisment: match {match_id} a esuat: {exc}")

        atomic_write_csv(
            pd.DataFrame(failed_matches, columns=["match_id", "error"]),
            selection_dir / "failed_matches.csv",
        )

    player_matches = pd.DataFrame(all_player_matches)
    if player_matches.empty:
        raise RuntimeError("Nu s-au obtinut randuri StatsBomb cu minute jucate.")
    for column in STATSBOMB_NUMERIC_FEATURES:
        if column not in player_matches.columns:
            player_matches[column] = 0
    player_matches[STATSBOMB_NUMERIC_FEATURES] = player_matches[STATSBOMB_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)

    output_file = STATSBOMB_OUTPUT_DIR / "statsbomb_player_match_stats.csv"
    atomic_write_csv(player_matches, output_file)
    available_features = [
        col for col in STATSBOMB_NUMERIC_FEATURES
        if col in player_matches.columns and pd.to_numeric(player_matches[col], errors="coerce").notna().any()
    ]
    print(f"StatsBomb: {len(player_matches):,} randuri jucator–meci; {len(available_features)} campuri numerice.")
    print("Salvat:", output_file)
    if len(player_matches) >= MIN_ROWS and len(available_features) >= MIN_FEATURES:
        print("PASS — datasetul StatsBomb trece pragurile de volum si features.")
    else:
        print("ATENTIE — adauga competitii/sezoane in STATSBOMB_SELECTIONS ca sa cresti datasetul.")
    return player_matches


## Ruleaza extragerea

Ruleaza aceasta celula dupa cele de configurare. Daca una dintre surse esueaza, cealalta continua. Verifica tabelul final si mesajele `PASS`. Pentru model, foloseste un singur fisier-sursa; nu concatena automat FBref cu StatsBomb.


In [7]:
run_results = []

if RUN_FBREF:
    try:
        fbref_dataset = extract_fbref()
        run_results.append({"source": "FBref", "status": "OK", "rows": len(fbref_dataset), "file": str(FBREF_OUTPUT_DIR / "fbref_player_match_stats.csv")})
    except Exception as exc:
        print(f"FBref nu a putut fi extras: {type(exc).__name__}: {exc}")
        run_results.append({"source": "FBref", "status": "FAILED", "rows": 0, "file": str(FBREF_OUTPUT_DIR / "fbref_player_match_stats.csv")})

if RUN_STATSBOMB:
    try:
        statsbomb_dataset = extract_statsbomb()
        run_results.append({"source": "StatsBomb", "status": "OK", "rows": len(statsbomb_dataset), "file": str(STATSBOMB_OUTPUT_DIR / "statsbomb_player_match_stats.csv")})
    except Exception as exc:
        print(f"StatsBomb nu a putut fi extras: {type(exc).__name__}: {exc}")
        run_results.append({"source": "StatsBomb", "status": "FAILED", "rows": 0, "file": str(STATSBOMB_OUTPUT_DIR / "statsbomb_player_match_stats.csv")})

run_summary = pd.DataFrame(run_results)
display(run_summary)


[11/16/26 01:45:16] INFO     Saving cached data to /Users/laur/Master Data                           _common.py:300
                             Science/sem1/IAPD/IAPD_project/notebooks/data/cache/fbref                             

FBref: folosesc 244 meciuri cu pagina deja in cache din 380 disponibile.


[11/16/26 01:45:19] INFO     [1/244] Retrieving game with id=3a6836b4                                  fbref.py:766

                    WARNING  /Users/laur/Master Data Science/sem1/SA/Tema                           warnings.py:112
                             1/.conda/lib/python3.12/site-packages/soccerdata/fbref.py:777:                        
                             FutureWarning: This search incorrectly ignores the root element, and                  
                             will be fixed in a future version.  If you rely on the current                        
                             behaviour, change it to ".//table"                                                    
                               html_table = tree.find("//table[@id='" +                                            
                             id_format.format(home_team["id"]) + "']")                                             
                                                                                                                   

                    WARNING  /Users/laur/Master Data Science/sem1/SA/Tema                           warnings.py:112
                             1/.conda/lib/python3.12/site-packages/soccerdata/fbref.py:788:                        
                             FutureWarning: This search incorrectly ignores the root element, and                  
                             will be fixed in a future version.  If you rely on the current                        
                             behaviour, change it to ".//table"                                                    
                               html_table = tree.find("//table[@id='" +                                            
                             id_format.format(away_team["id"]) + "']")                                             
                                                                                                                   

                    INFO     [2/244] Retrieving game with id=26a7f90c                                  fbref.py:766

... (728 outputs left out)

FBref: 7,286 randuri; 16 campuri statistice gasite.
Salvat: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project/notebooks/data/raw/fbref/fbref_player_match_stats.csv
PASS — datasetul FBref trece pragurile de volum si features.


,competition_id,season_id,competition_name,season_name,country_name
0,9,281,1. Bundesliga,2023/2024,Germany
1,9,27,1. Bundesliga,2015/2016,Germany
2,1267,107,African Cup of Nations,2023,Africa
3,16,4,Champions League,2018/2019,Europe
4,16,1,Champions League,2017/2018,Europe
5,16,2,Champions League,2016/2017,Europe
6,16,27,Champions League,2015/2016,Europe
7,16,26,Champions League,2014/2015,Europe
8,16,25,Champions League,2013/2014,Europe
9,16,24,Champions League,2012/2013,Europe


FIFA World Cup 2022: 64 meciuri
1/64 — match 3857276: checkpoint incarcat
2/64 — match 3857271: checkpoint incarcat
3/64 — match 3857296: checkpoint incarcat
4/64 — match 3857274: checkpoint incarcat
5/64 — match 3857255: checkpoint incarcat
6/64 — match 3857272: checkpoint incarcat
7/64 — match 3857278: checkpoint incarcat
8/64 — match 3857277: checkpoint incarcat
9/64 — match 3857273: checkpoint incarcat
10/64 — match 3857275: checkpoint incarcat
11/64 — match 3857261: checkpoint incarcat
12/64 — match 3857290: checkpoint incarcat
13/64 — match 3857298: checkpoint incarcat
14/64 — match 3857285: checkpoint incarcat
15/64 — match 3857282: checkpoint incarcat
16/64 — match 3857297: checkpoint incarcat
17/64 — match 3857294: checkpoint incarcat
18/64 — match 3857266: checkpoint incarcat
19/64 — match 3857284: checkpoint incarcat
20/64 — match 3857254: checkpoint incarcat
21/64 — match 3857265: checkpoint incarcat
22/64 — match 3857301: checkpoint incarcat
23/64 — match 3857268: checkpoi

,source,status,rows,file
0,FBref,OK,7286,/Users/laur/Master Data Science/sem1/IAPD/IAPD...
1,StatsBomb,OK,1995,/Users/laur/Master Data Science/sem1/IAPD/IAPD...


## Utilizare la predictie

Ambele CSV-uri au coloanele comune `player`, `date`, `Min`, `Gls`, `Ast`, `Sh`, `CrdY`, `CrdR`; StatsBomb adauga si xG, suturi pe poarta, pase si actiuni defensive. In notebookul de modelare alege una dintre cai:

- `data/raw/fbref/fbref_player_match_stats.csv`
- `data/raw/statsbomb/statsbomb_player_match_stats.csv`

Modelul construieste features din aparitiile anterioare si tinta din goluri + assisturi in meciul curent. Daca StatsBomb nu trece pragul de 1.000 de aparitii, adauga perechi competitie–sezon in `STATSBOMB_SELECTIONS`. Citeaza StatsBomb si foloseste atribuirea ceruta de repository cand prezinti rezultatele.
